In [ ]:
import numpy as np
import pandas as pd

In [ ]:
train = pd.read_csv('./toxic/train.csv')
train = train.sample(frac=0.1)
test = pd.read_csv('./toxic/test.csv')

In [ ]:
test = test.sample(frac=0.01)

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
classes = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

In [ ]:
y = train[classes].values

In [ ]:
features = 20000
max_length = 100

In [ ]:
from keras.preprocessing import text, sequence
from keras.layers import Dense, Embedding, Input, LSTM, Bidirectional, GlobalMaxPool1D, Dropout
from keras.models import Model


In [ ]:
tokenizer = text.Tokenizer(num_words=features)

In [ ]:
tokenizer

In [ ]:
list_sentences_test = test["comment_text"].fillna("CVxTz").values
list_sentences_train = train["comment_text"].fillna("CVxTz").values


In [ ]:
tokenizer.fit_on_texts(list(list_sentences_train))
list_tokenized_train = tokenizer.texts_to_sequences(list_sentences_train)

In [ ]:
list_tokenized_test = tokenizer.texts_to_sequences(list_sentences_test)

In [ ]:
X_t = sequence.pad_sequences(list_tokenized_train, maxlen=max_length)
X_te = sequence.pad_sequences(list_tokenized_test, maxlen=max_length)

In [ ]:
def get_model():
    embed_size = 128
    inp = Input(shape=(max_length, ))
    x = Embedding(features, embed_size)(inp)
    x = Bidirectional(LSTM(50, return_sequences=True))(x)
    x = GlobalMaxPool1D()(x)
    x = Dropout(0.1)(x)
    x = Dense(50, activation="relu")(x)
    x = Dropout(0.1)(x)
    x = Dense(6, activation="sigmoid")(x)
    model = Model(inputs=inp, outputs=x)
    model.compile(loss='binary_crossentropy',
                  optimizer='adam',
                  metrics=['accuracy'])

    return model

In [ ]:
model = get_model()
batch_size = 32
epochs = 2

In [ ]:
model.fit(X_t, y, batch_size=batch_size, epochs=epochs, validation_split=0.1)


In [ ]:
y_test = model.predict(X_te)

In [ ]:
X_te[0]

In [ ]:
y_test[0]

In [ ]:
test.head(40)

In [ ]:
[np.argmax(y) for y in y_test[:1200] if np.argmax(y) != 0]